# 03 · 내 문서 Grounding

**목표**: AI Core vector grounding 저장소의 문서를 검색하고, 검색 청크를 확인한 뒤 근거 기반 답변을 생성합니다.

공개 Help 저장소와 달리 data_repository_type을 vector로 바꾸고 collection ID를 지정합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: 내 문서 Grounding 질의</h2>
<p style="margin: 7px 0;">루트 .env의 GROUNDING_REPOSITORY_ID로 선택한 vector collection에 질문을 보냅니다. 아래 관리 UI에서 문서를 넣거나 삭제할 수도 있습니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import os
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '3_gen_ai_hub_services')]
from lab_common.config import load_settings
from src.documents.lab_core import (delete_document_by_id, list_document_ids, run_document_grounded_query, upload_document_text)

def run_my_docs_ui(user_query: str) -> str:
    try:
        settings = load_settings()
        return run_document_grounded_query(user_query, settings)
    except Exception as error:
        return f'내 문서 Grounding 호출 실패: {type(error).__name__}: {error}'

my_docs_ui = gr.Interface(
    fn=run_my_docs_ui,
    inputs=gr.Textbox(label='Question', lines=3),
    outputs=gr.Textbox(label='Grounded answer', lines=10),
    title='My Documents Grounding Playground',
    description='루트 .env의 GROUNDING_REPOSITORY_ID를 사용합니다.',
)
my_docs_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", height=900)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #f9ab00; background: #fef7e0; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #b06000; letter-spacing: .02em;">📁  저장소 관리</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">문서 업로드 및 제거</h2>
<p style="margin: 7px 0;">AI Core vector 저장소에는 텍스트 청크가 저장됩니다. 이 UI는 UTF-8 <code>.txt</code>와 <code>.md</code> 파일을 청크로 업로드합니다. 삭제는 즉시 적용되고 되돌릴 수 없습니다.</p>
  </div>
</div>


In [ ]:
# Custom Grounding Repository 문서 관리 UI (.txt/.md)
manager_settings = load_settings()
default_repository_id = os.getenv('GROUNDING_REPOSITORY_ID', '')

def _repository_id(value: str) -> str:
    repository_id = (value or default_repository_id).strip()
    if not repository_id:
        raise ValueError('GROUNDING_REPOSITORY_ID를 .env에 설정하거나 Repository ID를 입력하세요.')
    return repository_id

def _document_ids(repository_id: str) -> list[str]:
    return list_document_ids(manager_settings, repository_id=_repository_id(repository_id))

def refresh_documents(repository_id: str):
    try:
        ids = _document_ids(repository_id)
        summary = '등록된 문서가 없습니다.' if not ids else '등록된 document ID:\n' + '\n'.join(f'- {doc_id}' for doc_id in ids)
        return summary, gr.Dropdown(choices=ids, value=None)
    except Exception as error:
        return f'문서 목록 조회 실패: {type(error).__name__}: {error}', gr.Dropdown(choices=[], value=None)

def upload_document(repository_id: str, file_path: str | None):
    try:
        if not file_path:
            raise ValueError('업로드할 .txt 또는 .md 파일을 선택하세요.')
        path = Path(file_path)
        if path.suffix.lower() not in {'.txt', '.md'}:
            raise ValueError('현재는 UTF-8 .txt와 .md 파일만 지원합니다.')
        upload_document_text(manager_settings, repository_id=_repository_id(repository_id), filename=path.name, text=path.read_text(encoding='utf-8'))
        ids = _document_ids(repository_id)
        return f'{path.name} 업로드 요청을 완료했습니다. 인덱싱에는 잠시 시간이 걸릴 수 있습니다.', gr.Dropdown(choices=ids, value=None)
    except Exception as error:
        return f'업로드 실패: {type(error).__name__}: {error}', gr.Dropdown()

def remove_document(repository_id: str, document_id: str | None):
    try:
        delete_document_by_id(manager_settings, repository_id=_repository_id(repository_id), document_id=document_id or '')
        ids = _document_ids(repository_id)
        return f'{document_id} 문서를 삭제했습니다.', gr.Dropdown(choices=ids, value=None)
    except Exception as error:
        return f'삭제 실패: {type(error).__name__}: {error}', gr.Dropdown()

with gr.Blocks() as document_manager_ui:
    gr.Markdown('### Custom Grounding Repository 관리')
    repository_input = gr.Textbox(label='Repository ID', value=default_repository_id, placeholder='Grounding Management 카드의 UUID')
    with gr.Row():
        upload_file = gr.File(label='UTF-8 .txt 또는 .md 문서', file_types=['.txt', '.md'], type='filepath')
        upload_button = gr.Button('업로드', variant='primary')
    with gr.Row():
        document_selector = gr.Dropdown(label='삭제할 document ID', choices=[])
        refresh_button = gr.Button('문서 목록 새로고침')
        delete_button = gr.Button('선택 문서 삭제', variant='stop')
    status = gr.Textbox(label='상태', lines=5)
    refresh_button.click(refresh_documents, inputs=repository_input, outputs=[status, document_selector])
    upload_button.click(upload_document, inputs=[repository_input, upload_file], outputs=[status, document_selector])
    delete_button.click(remove_document, inputs=[repository_input, document_selector], outputs=[status, document_selector])

document_manager_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1')


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 한 번 실행해 결과를 확인한다.</li>
<li>결과가 기대와 다르면 환경 설정과 모델/저장소 이름을 먼저 점검한다.</li>
<li>아래 코드는 같은 기능의 구성 요소를 자세히 살펴보는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 사용하는 구성과 동작을 단계별로 확인하는 자료입니다. 기본 실습에서는 실행하지 않아도 됩니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. AI Core REST와 SDK 준비</h2>
<p style="margin: 7px 0;">이 노트북은 REST로 collection과 검색 결과를 확인하고 SDK Orchestration으로 최종 답변을 생성합니다. 모두 같은 .env의 AI Core 인증을 사용합니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '3_gen_ai_hub_services')]
from lab_common.config import load_settings
from src.documents.lab_core import find_document_repository, run_document_grounded_query, search_document_chunks

settings = load_settings()
print("=" * 56)
print("Step 5 · 내 문서 그라운딩")
print("=" * 56)

base = os.getenv("AICORE_BASE_URL").rstrip("/")
h = api_headers(settings)
DG = f"{base}/v2/lm/document-grounding"

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 저장소 선택</h2>
<p style="margin: 7px 0;">GROUNDING_REPOSITORY_ID가 있으면 그 저장소를 사용합니다. 없으면 문서가 들어 있는 첫 collection을 읽기 전용으로 찾습니다. 교육 환경에서는 강사가 공유한 ID를 .env에 지정하는 편이 가장 재현성이 좋습니다.</p>
  </div>
</div>


In [ ]:
# 1) 저장소 id 결정 (.env 우선, 없으면 문서가 있는 첫 collection)
repo_id = find_document_repository(settings)
print("  저장소 id:", repo_id)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 질문 설정</h2>
<p style="margin: 7px 0;">Q 환경변수로 질문을 바꿀 수 있습니다. 질문은 저장소 문서의 실제 내용과 관련 있게 작성해야 검색 결과를 관찰하기 쉽습니다.</p>
  </div>
</div>


In [ ]:
# 2) 질문 — 저장소 내용에 맞춰 (procurement policy 가정 + 일반 질문 둘 다 시도)
QUESTION = os.getenv("Q", "What does this document say? Summarize the key points.")

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 검색된 청크 먼저 보기</h2>
<p style="margin: 7px 0;">LLM 답변 전에 retrieval/search 결과와 score를 확인합니다. 이는 모델 답변의 근거를 살피는 단계입니다. 점수 절대값보다 질문을 바꿨을 때 어떤 청크가 선택되는지 비교하세요.</p>
  </div>
</div>


In [ ]:
# 2a) 먼저 검색된 청크 보기 (retrieval/search)
print(f"\n[1] 검색된 청크 보기 — '{QUESTION}'")
chunks = search_document_chunks(QUESTION, settings, repository_id=repo_id)
found = 0
for ch in chunks:
    found += 1
    score = ch.get('searchScores', {}).get('aggregatedScore', {}).get('value', 0)
    print(f"  [{found}] score={score:.3f}  {ch.get('content', '')[:120]}")
if not found:
    print("  (검색된 청크 없음)")

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. Vector Grounding으로 답변 생성</h2>
<p style="margin: 7px 0;">마지막 단계는 공개 Grounding과 거의 같습니다. 필터만 vector와 선택한 collection ID로 바꾸어 내 문서 근거를 prompt에 주입합니다.</p>
  </div>
</div>


In [ ]:
# 3) 그라운딩 질의 — 모듈 4와 '필터만' 다름
print(f"\n[2] 그라운딩 LLM 답변 — '{QUESTION}'")
answer = run_document_grounded_query(QUESTION, settings, repository_id=repo_id)
print("  응답:", answer[:500])

print("\n✅ Step 5 완료 — 모듈 4 대비 '필터만' 바꿔 내 문서 기반 답변.")